# Lembar Kerja Mahasiswa (LKM)
## Latihan Mandiri — Pertemuan 07: Regresi Linear (Bagian 2)

Lembar kerja ini menuntun Anda mengerjakan **kedelapan soal Latihan Mandiri** pada bagian Penutup modul `Pertemuan07_Regresi_Linear_2.ipynb`, yang mencakup geometri regresi linear, regularisasi Ridge dan Lasso, serta ill-conditioning.

---

### Identitas Mahasiswa

*Klik dua kali sel ini untuk mengedit, isi titik-titik di bawah, lalu tekan `Shift + Enter`.*

| | |
|---|---|
| **Nama** | ............................................................ |
| **NIM** | ............................................................ |
| **Kelas / Rombel** | ............................................................ |
| **Nama Dosen** | ............................................................ |
| **Tanggal Pengumpulan** | ............................................................ |

---

### Capaian yang Diukur

Setelah menyelesaikan lembar kerja ini, Anda diharapkan mampu:

1. Membuktikan secara numerik bahwa residual regresi kuadrat terkecil tegak lurus terhadap ruang kolom $\mathbb{X}$.
2. Menelusuri perubahan solusi Ridge dan Lasso seiring $\lambda$ berubah, serta menjelaskan mengapa keduanya berbeda perilaku.
3. Mengaitkan derajat model dengan letak $\lambda$ terbaik.
4. Menghitung dan menafsirkan jumlah koefisien yang menyusut ke nol pada Ridge dibanding Lasso.
5. Menunjukkan pengaruh derajat kolinearitas terhadap bilangan kondisi dan kestabilan bobot.
6. Menjelaskan pertukaran antara kestabilan koefisien dan kedekatannya dengan bobot sebenarnya.


---
## Petunjuk Pengerjaan

**Baca bagian ini sebelum mulai.**

1. Lembar kerja ini **berdiri sendiri** (*self-contained*): bagian Persiapan di bawah membangun ulang seluruh fungsi dan variabel yang dipakai kedelapan latihan, sehingga Anda tidak perlu membuka notebook utama untuk menjalankannya.
2. Meskipun begitu, **sangat dianjurkan** membuka `Pertemuan07_Regresi_Linear_2.ipynb` di jendela terpisah sambil mengerjakan, terutama untuk melihat versi **interaktif** dari visualisasi Ridge dan Lasso (Bagian 3) serta animasi regularisasi (Bagian 4) — lembar kerja ini memakai versi yang lebih ringkas agar hasilnya dapat dicatat dalam tabel.
3. Jalankan sel kode **berurutan dari atas ke bawah**.
4. Sel yang berisi tanda `____` atau komentar `# TODO` **harus Anda lengkapi sendiri**.
5. Setiap latihan memiliki **tabel hasil pengamatan** dan **kotak jawaban** markdown. Klik dua kali untuk mengedit, lalu tekan `Shift + Enter`.

> **Penting.** Nilai `random_state` diturunkan dari NIM Anda, sehingga angka hasil percobaan setiap mahasiswa akan sedikit berbeda. Pola dan kesimpulannya akan tetap sama.

### Perkiraan waktu

| Bagian | Perkiraan |
|---|---|
| Persiapan | 5 menit |
| Latihan 1 (geometri) | 15 menit |
| Latihan 2–3 (Ridge dan Lasso) | 35 menit |
| Latihan 4–5 (derajat dan koefisien) | 30 menit |
| Latihan 6–8 (ill-conditioning) | 30 menit |
| Kesimpulan dan ekspor PDF | 10 menit |

### Cara menyimpan sebagai PDF

* **JupyterLab:** `File` → `Save and Export Notebook As...` → `HTML`, lalu buka berkas HTML di browser dan cetak (`Ctrl + P`) dengan tujuan **Save as PDF**.
* **Google Colab:** `File` → `Print` → tujuan **Save as PDF**.
* Beri nama berkas: `LKM_Latihan07_NIM_NamaLengkap.pdf`


---
## Persiapan

Bagian ini membangun ulang fungsi dan variabel dari notebook utama. **Sudah lengkap — jalankan saja**, kecuali sel P.2 yang meminta NIM Anda.

### P.1 Memuat pustaka

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

print("Semua pustaka berhasil dimuat.")

### P.2 Mengisi NIM Anda

In [ ]:
# TODO (a): ganti dengan tiga digit terakhir NIM Anda, misalnya 137
NIM_3_DIGIT = ____

SEED = int(NIM_3_DIGIT)
print("random_state pribadi Anda (SEED) =", SEED)

### P.3 Membangun ulang: Geometri Regresi Linear

Kode berikut membangun kembali visualisasi tiga dimensi dari Bagian 1 notebook utama: dua vektor kolom ($x_1, x_2$), vektor prediksi $Y = \mathbb{X}\mathbf{w}$, target $\mathbf{t}$, dan residual $\mathbf{e}$.

> **Anda tidak perlu memahami detail kode ini.** Fokuslah pada hasil visualnya dan pada Latihan 1 setelahnya.

In [ ]:
rng = np.random.default_rng(SEED)

SCALE = 2.0
x1_geo = np.array([3.0, 0.4, 0.6]) * SCALE
x2_geo = np.array([1.2, 2.4, -0.5]) * SCALE
bobot1, bobot2 = 1.5, 0.9
tinggi_residual = 2.2 * SCALE

def unit(v):
    n = np.linalg.norm(v)
    return v / n if n else v

def gram_schmidt(a, b):
    u1 = unit(a)
    b_perp = b - np.dot(b, u1) * u1
    u2 = unit(b_perp)
    return u1, u2

O = np.zeros(3)
Y_geo = bobot1 * x1_geo + bobot2 * x2_geo
u_hat, v_hat = gram_schmidt(x1_geo, x2_geo)
n_hat = unit(np.cross(u_hat, v_hat))
t_geo = Y_geo + tinggi_residual * n_hat

def arrow_traces(start, end, name, color, width=12, head=1.6):
    vec = end - start
    L = np.linalg.norm(vec)
    tip_base = end - max(0.9 * head, 0.02 * L) * (vec / L)
    line = go.Scatter3d(x=[start[0], tip_base[0]], y=[start[1], tip_base[1]], z=[start[2], tip_base[2]],
                        mode="lines", line=dict(width=width, color=color), name=name)
    cone = go.Cone(x=[end[0]], y=[end[1]], z=[end[2]], u=[vec[0]], v=[vec[1]], w=[vec[2]],
                   sizemode="absolute", sizeref=head, anchor="tip", showscale=False,
                   colorscale=[[0, color], [1, color]], showlegend=False)
    return line, cone

traces = []
for a, b, nm, cl in [(O, x1_geo, "x1 (kolom 1)", "darkgreen"), (O, x2_geo, "x2 (kolom 2)", "darkgreen"),
                     (O, Y_geo, "Y = Xw", "black"), (O, t_geo, "t (target)", "#d62728"),
                     (Y_geo, t_geo, "e = t - Y", "#ff9800")]:
    line, cone = arrow_traces(a, b, nm, cl)
    traces += [line, cone]

fig_geo = go.Figure(data=traces)
fig_geo.update_layout(title="Geometri Regresi Linear (Persiapan Latihan 1)",
                      width=700, height=600, scene=dict(aspectmode="cube"))
fig_geo.show()

print("Vektor yang tersedia untuk Latihan 1: x1_geo, x2_geo, Y_geo, t_geo")

### P.4 Membangun ulang: Ridge dan Lasso pada Ruang Parameter

Kode berikut membangun kembali fungsi dari Bagian 3 notebook utama: `plot_E_D` (kontur galat data), `ridge_solution`, dan `lasso_solution`.

In [ ]:
w0_par = np.array([3.2, 0.9])
theta = np.deg2rad(25)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
A_par = R @ np.diag([6.0, 1.0]) @ R.T * 2.5


def plot_E_D():
    x = np.linspace(-10.0, 10.0, 300)
    y = np.linspace(-10.0, 10.0, 300)
    X, Y = np.meshgrid(x, y)
    U = np.stack([X - w0_par[0], Y - w0_par[1]], axis=-1)
    Z_ED = 0.5 * np.einsum("...i,...i", U, U @ A_par)
    fig = go.Figure()
    fig.add_trace(go.Contour(x=x, y=y, z=Z_ED, colorscale="Blues", reversescale=True,
                             showscale=False, opacity=0.9, ncontours=20))
    fig.add_trace(go.Scatter(x=[w0_par[0]], y=[w0_par[1]], mode="markers",
                             marker=dict(symbol="star", size=14, color="crimson"),
                             name="ŵ(λ=0)"))
    fig.update_layout(width=550, height=550, template="plotly_white",
                      xaxis_title="w1", yaxis_title="w2",
                      xaxis=dict(scaleanchor="y", range=[w0_par[0]-10, w0_par[0]+10]),
                      yaxis=dict(range=[w0_par[1]-10, w0_par[1]+10]))
    return fig


def ridge_solution(lmbda, A=A_par, w0=w0_par):
    """Solusi tertutup Ridge: w = (A + lambda*I)^{-1} A w0."""
    w_hat = np.linalg.solve(A + lmbda * np.eye(2), A @ w0)
    ED = 0.5 * (w_hat - w0) @ A @ (w_hat - w0)
    EW = 0.5 * (w_hat @ w_hat)
    return w_hat, ED, EW, ED + lmbda * EW


def lasso_objective(w, A, w0, lmbda):
    ED = 0.5 * (w - w0).T @ A @ (w - w0)
    EW = np.sum(np.abs(w))
    return ED + lmbda * EW


def lasso_solution(lmbda, A=A_par, w0=w0_par):
    """Solusi Lasso, diselesaikan secara numerik karena tidak punya rumus tertutup."""
    hasil = minimize(lasso_objective, np.zeros(2), args=(A, w0, lmbda), method="Nelder-Mead", tol=1e-6)
    w_hat = hasil.x
    ED = 0.5 * (w_hat - w0).T @ A @ (w_hat - w0)
    EW = np.sum(np.abs(w_hat))
    return w_hat, ED, EW, ED + lmbda * EW


print("Fungsi siap: plot_E_D(), ridge_solution(lmbda), lasso_solution(lmbda)")
print("Bobot tanpa regularisasi w0 =", w0_par)

### P.5 Membangun Ulang: Evaluasi Regularisasi pada Model Polinomial

Fungsi berikut adalah **versi ringkas** dari `plot_regularized_model_performance` pada Bagian 4 notebook utama. Alih-alih menghasilkan animasi, fungsi ini langsung mengembalikan **tabel** MSE dan koefisien untuk setiap $\lambda$ — lebih mudah dianalisis dalam lembar kerja ini.

> Data yang dibangkitkan sama persis konsepnya dengan aslinya: target sebenarnya $2\sin(2\pi x)$ dengan derau, dicocokkan memakai polinomial derajat tinggi.

In [ ]:
def latih_dan_evaluasi(model_class, derajat, daftar_lambda, rasio_uji=0.85, seed=SEED):
    """Melatih model teregularisasi untuk beberapa nilai lambda sekaligus.

    Mengembalikan dua DataFrame: (1) MSE latih & uji per lambda, (2) koefisien per lambda.
    """
    rng_lokal = np.random.default_rng(seed)
    X = np.linspace(0, 1, 40).reshape(-1, 1)
    y_true = 2 * np.sin(2 * np.pi * X).ravel()
    y = y_true + rng_lokal.normal(0, 0.7, X.shape[0])

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=rasio_uji, random_state=seed)

    poly = PolynomialFeatures(degree=derajat, include_bias=False)
    Xtr_p = poly.fit_transform(X_tr)
    Xte_p = poly.transform(X_te)

    baris_mse, koef = [], {}
    for lam in daftar_lambda:
        model = model_class(alpha=lam, fit_intercept=True, max_iter=50000)
        model.fit(Xtr_p, y_tr)
        mse_tr = mean_squared_error(y_tr, model.predict(Xtr_p))
        mse_te = mean_squared_error(y_te, model.predict(Xte_p))
        baris_mse.append({"lambda": lam, "mse_latih": mse_tr, "mse_uji": mse_te})
        koef[f"lambda={lam:g}"] = model.coef_.ravel()

    tabel_mse = pd.DataFrame(baris_mse)
    tabel_koef = pd.DataFrame(koef)
    tabel_koef.index = [f"w{i+1}" for i in range(len(tabel_koef))]
    return tabel_mse, tabel_koef


DAFTAR_LAMBDA = [100, 35, 20, 1, 0.2, 0.05, 0.01, 0.003, 0.0009]

# Latihan 4 dan 5 memakai data dengan SEED TETAP (bukan SEED pribadi Anda).
# Ini disengaja: kedua latihan itu bersifat demonstratif, sehingga datanya
# dibuat sama untuk semua mahasiswa agar polanya konsisten dan dapat dibandingkan.
SEED_DEMO = 29

print("Fungsi siap: latih_dan_evaluasi(model_class, derajat, daftar_lambda)")

---
# Latihan Mandiri 1 — Geometri: Membuktikan Ketegaklurusan Residual

**Soal asli:** *Pada grafik geometri tiga dimensi, putar tampilannya sampai bidang terlihat dari samping. Buktikan secara numerik bahwa residual tegak lurus terhadap kedua vektor kolom dengan memeriksa bahwa hasil kali titiknya mendekati nol.*

### 1.1 Mengamati secara visual

Putar grafik pada sel P.3 di atas dengan tetikus sampai bidang hijau ($\text{span}(\mathbb{X})$) terlihat sebagai sebuah **garis tipis** (dilihat dari samping). Pada sudut pandang itu, panah oranye ($\mathbf{e}$) seharusnya terlihat tegak lurus terhadap bidang tersebut.

### 1.2 Membuktikan secara numerik

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): hitung vektor residual e = t - Y
e_geo = t_geo - ____

# TODO (b): hitung hasil kali titik (dot product) antara e dan masing-masing vektor kolom
dot_x1 = np.dot(e_geo, ____)
dot_x2 = np.dot(____, x2_geo)

print("e . x1 =", dot_x1)
print("e . x2 =", dot_x2)
print("Norma e (panjang residual):", np.linalg.norm(e_geo))
print("\nKeduanya mendekati nol?", np.isclose(dot_x1, 0, atol=1e-8) and np.isclose(dot_x2, 0, atol=1e-8))

### 1.3 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| $\mathbf{e} \cdot x_1$ | ...... |
| $\mathbf{e} \cdot x_2$ | ...... |
| Norma $\|\mathbf{e}\|$ | ...... |
| Apakah keduanya \"mendekati nol\"? | ...... |

### 1.4 Pertanyaan Analisis

**1a.** Apakah kedua hasil kali titik tersebut **persis** nol, atau hanya sangat kecil (misalnya $10^{-14}$)? Jelaskan mengapa demikian.

> *Jawaban Anda:*
>
> ......

**1b.** Bandingkan besar `dot_x1`/`dot_x2` dengan `norma e`. Mengapa perbandingan ini lebih bermakna daripada sekadar melihat apakah hasil kali titiknya "kecil"?

> *Jawaban Anda:*
>
> ......

**1c.** Hubungkan temuan ini dengan syarat $\mathbb{X}^T\mathbf{e} = 0$ pada penurunan persamaan normal. Mengapa syarat itu setara dengan "$\mathbf{e}$ tegak lurus terhadap setiap kolom $\mathbb{X}$"?

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 2 — Ridge: Menelusuri Pengaruh λ

**Soal asli:** *Pada visualisasi Ridge, geser λ dari 0 sampai 15. Bagaimana perubahan jari-jari lingkaran, dan ke arah mana solusinya bergerak?*

### 2.1 Menjalankan Ridge untuk banyak nilai λ

Lengkapi bagian bertanda `____`.

In [ ]:
lambdas_ridge = np.linspace(0, 15, 16)
hasil_ridge = []

for lam in lambdas_ridge:
    # TODO (a): panggil ridge_solution untuk nilai lam ini
    w_hat, ED, EW, J = ridge_solution(____)
    jari_jari = np.linalg.norm(w_hat)   # c(lambda) = ||w||_2
    hasil_ridge.append({"lambda": lam, "w1": w_hat[0], "w2": w_hat[1],
                        "jari_jari": jari_jari, "E_D": ED})

tabel_ridge = pd.DataFrame(hasil_ridge).round(4)
display(tabel_ridge)

### 2.2 Menggambar lintasan solusi

In [ ]:
fig = plot_E_D()
fig.add_trace(go.Scatter(x=tabel_ridge["w1"], y=tabel_ridge["w2"], mode="lines+markers",
                         line=dict(color="teal", width=3), marker=dict(size=6),
                         name="Lintasan ŵ(λ)"))
fig.add_trace(go.Scatter(x=[w0_par[0]], y=[w0_par[1]], mode="markers",
                         marker=dict(symbol="x", size=12, color="black"), name="Titik asal referensi"))
fig.update_layout(title="Lintasan Solusi Ridge untuk λ = 0 sampai 15")
fig.show()

print("Jari-jari pada λ=0  :", round(tabel_ridge['jari_jari'].iloc[0], 4))
print("Jari-jari pada λ=15 :", round(tabel_ridge['jari_jari'].iloc[-1], 4))

### 2.3 Tabel Hasil Pengamatan

| λ | $w_1$ | $w_2$ | Jari-jari $\|\mathbf{w}\|$ |
|---|---|---|---|
| 0 | ...... | ...... | ...... |
| 5 | ...... | ...... | ...... |
| 10 | ...... | ...... | ...... |
| 15 | ...... | ...... | ...... |

### 2.4 Pertanyaan Analisis

**2a.** Bagaimana perubahan jari-jari lingkaran (kolom "Jari-jari") seiring λ membesar dari 0 ke 15?

> *Jawaban Anda:*
>
> ......

**2b.** Ke arah mana titik solusi $\hat{\mathbf{w}}(\lambda)$ bergerak? Apakah pernah mencapai tepat $(0,0)$ pada λ = 15?

> *Jawaban Anda:*
>
> ......

**2c.** Perhatikan bentuk lintasannya pada grafik (garis lurus atau melengkung?). Mengapa lintasan Ridge cenderung melengkung mengikuti bentuk kontur galat, bukan garis lurus menuju titik asal?

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 3 — Lasso: Mencari Titik Sudut

**Soal asli:** *Ulangi latihan nomor 2 pada visualisasi Lasso. Pada nilai λ berapa solusinya mulai menyentuh sudut belah ketupat? Apa arti kejadian itu bagi koefisiennya?*

### 3.1 Menjalankan Lasso untuk banyak nilai λ

Lengkapi bagian bertanda `____`.

In [ ]:
lambdas_lasso = np.linspace(0, 15, 31)
hasil_lasso = []

for lam in lambdas_lasso:
    # TODO (a): panggil lasso_solution untuk nilai lam ini
    w_hat, ED, EW, J = lasso_solution(____)
    hasil_lasso.append({"lambda": lam, "w1": w_hat[0], "w2": w_hat[1],
                        "norma_l1": np.sum(np.abs(w_hat))})

tabel_lasso = pd.DataFrame(hasil_lasso).round(4)

# TODO (b): cari baris PERTAMA ketika w1 ATAU w2 sudah sangat dekat ke nol (< 0.05)
menyentuh_sudut = tabel_lasso[(tabel_lasso["w1"].abs() < ____) | (tabel_lasso["w2"].abs() < 0.05)]

print("Lima baris pertama:"); display(tabel_lasso.head())
if len(menyentuh_sudut) > 0:
    print("\nλ pertama kali menyentuh sudut:", menyentuh_sudut.iloc[0]['lambda'])
    display(menyentuh_sudut.head(1))
else:
    print("\nBelum ada koefisien yang mencapai nol pada rentang λ ini.")

### 3.2 Menggambar lintasan solusi Lasso

In [ ]:
fig = plot_E_D()
fig.add_trace(go.Scatter(x=tabel_lasso["w1"], y=tabel_lasso["w2"], mode="lines+markers",
                         line=dict(color="darkorange", width=3), marker=dict(size=5),
                         name="Lintasan ŵ(λ) — Lasso"))
fig.update_layout(title="Lintasan Solusi Lasso untuk λ = 0 sampai 15")
fig.show()

### 3.3 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| λ pertama kali salah satu koefisien ≈ 0 | ...... |
| Koefisien mana yang lebih dulu menjadi nol, $w_1$ atau $w_2$? | ...... |
| Nilai $w_1, w_2$ pada λ = 15 | ...... |

### 3.4 Pertanyaan Analisis

**3a.** Pada λ berapa solusi Lasso mulai "menempel" di salah satu sumbu (artinya satu koefisien tepat nol)? Bandingkan dengan lintasan Ridge pada Latihan 2 — apakah Ridge pernah menempel di sumbu?

> *Jawaban Anda:*
>
> ......

**3b.** Jelaskan **mengapa** perbedaan ini terjadi, dikaitkan dengan bentuk daerah kendala (lingkaran vs belah ketupat) yang dibahas pada Bagian 3 notebook utama.

> *Jawaban Anda:*
>
> ......

**3c.** Bila $w_2$ sudah bernilai tepat nol, apa arti praktisnya bagi model? Fitur ke berapa yang sesungguhnya sedang "dibuang" dari model?

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 4 — Pengaruh Derajat Polinomial terhadap λ Terbaik

**Soal asli:** *Pada Bagian 4, jalankan `plot_regularized_model_performance` dengan `degree=3` lalu `degree=12`. Bagaimana pengaruh derajat polinomial terhadap letak λ terbaik?*

> Kita memakai `latih_dan_evaluasi` (versi ringkas dari `plot_regularized_model_performance`) agar hasilnya dapat dibandingkan dalam tabel.

### 4.1 Menjalankan untuk dua derajat berbeda

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): jalankan untuk derajat = 3
mse_d3, koef_d3 = latih_dan_evaluasi(Ridge, derajat=____, daftar_lambda=DAFTAR_LAMBDA, seed=SEED_DEMO)

# TODO (b): jalankan untuk derajat = 12
mse_d12, koef_d12 = latih_dan_evaluasi(Ridge, derajat=____, daftar_lambda=DAFTAR_LAMBDA, seed=SEED_DEMO)

lam_terbaik_d3 = mse_d3.loc[mse_d3["mse_uji"].idxmin(), "lambda"]
lam_terbaik_d12 = mse_d12.loc[mse_d12["mse_uji"].idxmin(), "lambda"]

print("=== Derajat 3 ==="); display(mse_d3.round(4))
print("λ terbaik (derajat 3) :", lam_terbaik_d3)
print("\n=== Derajat 12 ==="); display(mse_d12.round(4))
print("λ terbaik (derajat 12):", lam_terbaik_d12)

### 4.2 Menggambar perbandingan kurva MSE uji

In [ ]:
import matplotlib.pyplot as plt
plt.plot(mse_d3["lambda"], mse_d3["mse_uji"], "o-", label="Derajat 3")
plt.plot(mse_d12["lambda"], mse_d12["mse_uji"], "o-", label="Derajat 12")
plt.xscale("log"); plt.xlabel("λ (skala log)"); plt.ylabel("MSE Uji")
plt.title("Pengaruh Derajat terhadap Kurva MSE Uji vs λ")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 4.3 Tabel Hasil Pengamatan

| Derajat | λ terbaik | MSE uji pada λ terbaik | MSE uji pada λ = 100 (regularisasi terkuat) |
|---|---|---|---|
| 3 | ...... | ...... | ...... |
| 12 | ...... | ...... | ...... |

### 4.4 Pertanyaan Analisis

**4a.** Model derajat mana yang membutuhkan λ **lebih besar** untuk mencapai MSE uji terkecil? Menurut Anda, mengapa demikian?

> *Jawaban Anda:*
>
> ......

**4b.** Model derajat 3 punya sedikit parameter, model derajat 12 punya banyak parameter. Kaitkan jumlah parameter itu dengan seberapa besar regularisasi yang dibutuhkan masing-masing.

> *Jawaban Anda:*
>
> ......

**4c.** Bila derajatnya terlalu rendah (model sudah *underfitting* secara bawaan), apakah regularisasi (menambah λ) dapat memperbaikinya? Jelaskan alasan Anda.

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 5 — Menghitung Koefisien yang Menjadi Nol

**Soal asli:** *Amati tabel koefisien hasil Ridge dan hasil Lasso. Hitung berapa banyak koefisien yang bernilai tepat nol pada masing-masing. Apakah hasilnya sesuai dengan teori?*

> Kita membandingkan pada $\lambda = 0{,}05$, karena pada nilai yang jauh lebih besar Lasso sudah membuang **seluruh** koefisien sehingga tidak ada lagi yang bisa dibandingkan.

### 5.1 Membandingkan tabel koefisien Ridge dan Lasso

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): jalankan latih_dan_evaluasi dengan Lasso, derajat 8, memakai DAFTAR_LAMBDA yang sama
mse_ridge8, koef_ridge8 = latih_dan_evaluasi(Ridge, derajat=8, daftar_lambda=DAFTAR_LAMBDA, seed=SEED_DEMO)
mse_lasso8, koef_lasso8 = latih_dan_evaluasi(____, derajat=8, daftar_lambda=DAFTAR_LAMBDA, seed=SEED_DEMO)

# Kita bandingkan pada lambda = 0.05 (bukan yang terbesar): pada nilai inilah
# perbedaan Ridge dan Lasso paling kentara pada data ini.
lambda_pembanding = "lambda=0.05"

print("Koefisien Ridge pada", lambda_pembanding, ":")
display(koef_ridge8[[lambda_pembanding]].round(4))

print("\nKoefisien Lasso pada", lambda_pembanding, ":")
display(koef_lasso8[[lambda_pembanding]].round(4))

# TODO (b): hitung berapa banyak koefisien yang nilai mutlaknya di bawah ambang batas 1e-3
ambang = 1e-3
jumlah_nol_ridge = (koef_ridge8[lambda_pembanding].abs() < ____).sum()
jumlah_nol_lasso = (koef_lasso8[lambda_pembanding].abs() < ambang).sum()

print(f"\nJumlah koefisien mendekati nol pada Ridge : {jumlah_nol_ridge} dari {len(koef_ridge8)}")
print(f"Jumlah koefisien mendekati nol pada Lasso : {jumlah_nol_lasso} dari {len(koef_lasso8)}")

### 5.2 Tabel Hasil Pengamatan

| Model | Jumlah koefisien ≈ 0 pada λ = 0,05 | Total koefisien |
|---|---|---|
| Ridge | ...... | ...... |
| Lasso | ...... | ...... |

### 5.3 Pertanyaan Analisis

**5a.** Model mana yang memiliki lebih banyak koefisien bernilai nol pada $\lambda=0{,}05$ ini? Apakah hasil ini sesuai dengan teori pada Bagian 3 notebook utama?

> *Jawaban Anda:*
>
> ......

**5b.** Perhatikan nilai koefisien Ridge yang **tidak** nol (meski kecil). Apakah nilainya benar-benar nol atau hanya sangat kecil? Bandingkan dengan koefisien Lasso yang nol.

> *Jawaban Anda:*
>
> ......

**5c.** Bila Anda memiliki 100 fitur dan hanya 10 yang benar-benar relevan, model mana — Ridge atau Lasso — yang lebih membantu Anda **mengidentifikasi** fitur mana saja yang relevan itu? Jelaskan.

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 6 — Ill-Conditioning: Mengubah Derajat Kolinearitas

**Soal asli:** *Pada Bagian 5, ubah `1e-4` menjadi `1e-2` lalu `1e-8`, dan amati perubahan bilangan kondisinya. Pada nilai berapa bobot hasil persamaan normal mulai kacau?*

### 6.1 Menuliskan fungsi ill-conditioning sebagai fungsi yang dapat dipakai ulang

Lengkapi bagian bertanda `____`.

In [ ]:
def uji_kolinearitas(gangguan, seed=SEED):
    """Membangun dua fitur yang hampir kolinear, lalu membandingkan
    persamaan normal biasa dengan Ridge.

    Argumen:
        gangguan: seberapa jauh x2 menyimpang dari x1 (semakin kecil, semakin kolinear).
    """
    rng_lokal = np.random.default_rng(seed)
    N, D = 500, 2
    X = np.ones((N, D + 1))
    x1 = rng_lokal.standard_normal(N)
    # TODO (a): buat x2 hampir identik dengan x1, disimpangkan sebesar `gangguan`
    x2 = x1 + ____ * rng_lokal.standard_normal(N)
    X[:, 1] = x1
    X[:, 2] = x2

    t = 3 + 2 * x1 - 1 * x2 + 0.1 * rng_lokal.standard_normal(N)

    XtX = X.T @ X
    Xty = X.T @ t
    w_ne = np.linalg.solve(XtX, Xty)

    # TODO (b): hitung nilai singular dengan SVD, lalu bilangan kondisi kappa = sigma_max/sigma_min
    _, s, _ = np.linalg.svd(X, full_matrices=False)
    kappa = s.____() / s.____()

    w_ridge = np.linalg.solve(XtX + 1e-2 * np.eye(D + 1), Xty)

    return {"gangguan": gangguan, "kappa_XtX": kappa ** 2,
           "w_ne": w_ne, "w_ridge": w_ridge}


hasil_6 = [uji_kolinearitas(g) for g in [1e-2, 1e-4, 1e-8]]
for h in hasil_6:
    print(f"gangguan={h['gangguan']:.0e}  |  cond(XtX)={h['kappa_XtX']:.3e}")
    print("   w_ne   :", np.round(h['w_ne'], 3))
    print("   w_ridge:", np.round(h['w_ridge'], 3))

### 6.2 Tabel Hasil Pengamatan

*Bobot sebenarnya adalah $[3,\ 2,\ -1]$.*

| Gangguan | Bilangan kondisi cond($\mathbb{X}^T\mathbb{X}$) | $\mathbf{w}$ dari persamaan normal | $\mathbf{w}$ dari Ridge |
|---|---|---|---|
| $10^{-2}$ | ...... | ...... | ...... |
| $10^{-4}$ | ...... | ...... | ...... |
| $10^{-8}$ | ...... | ...... | ...... |

### 6.3 Pertanyaan Analisis

**6a.** Bagaimana bilangan kondisi berubah ketika gangguan diperkecil dari $10^{-2}$ menjadi $10^{-8}$? Apakah perubahannya proporsional (misalnya sepuluh kali lebih kecil menyebabkan bilangan kondisi sepuluh kali lebih besar), atau jauh lebih drastis?

> *Jawaban Anda:*
>
> ......

**6b.** Pada gangguan berapa bobot hasil persamaan normal mulai terlihat "kacau" (nilainya jauh dari $[3,2,-1]$ dan/atau sangat besar)? Bandingkan dengan bobot Ridge pada gangguan yang sama.

> *Jawaban Anda:*
>
> ......

**6c.** Mengapa bobot Ridge tetap masuk akal pada ketiga tingkat gangguan, padahal Ridge sama sekali tidak "tahu" bahwa datanya hampir kolinear?

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 7 — Pertukaran Kestabilan vs Kedekatan dengan Bobot Sebenarnya

**Soal asli:** *Masih pada Bagian 5, coba beberapa nilai `lam` untuk Ridge: `1e-6`, `1e-2`, dan `1`. Adakah pertukaran antara kestabilan koefisien dan kedekatannya dengan bobot sebenarnya?*

### 7.1 Mencoba beberapa kekuatan regularisasi

Lengkapi bagian bertanda `____`. Kita memakai gangguan tetap $10^{-4}$ (kolinearitas sedang) agar efek λ terlihat jelas.

In [ ]:
W_SEBENARNYA_7 = np.array([3.0, 2.0, -1.0])

def uji_lambda_ridge(lam, gangguan=1e-4, seed=SEED):
    rng_lokal = np.random.default_rng(seed)
    N, D = 500, 2
    X = np.ones((N, D + 1))
    x1 = rng_lokal.standard_normal(N)
    x2 = x1 + gangguan * rng_lokal.standard_normal(N)
    X[:, 1], X[:, 2] = x1, x2
    t = 3 + 2 * x1 - 1 * x2 + 0.1 * rng_lokal.standard_normal(N)

    XtX = X.T @ X
    Xty = X.T @ t

    # TODO (a): selesaikan Ridge untuk nilai lam ini: (XtX + lam*I)^{-1} Xty
    w_ridge = np.linalg.solve(XtX + ____ * np.eye(D + 1), Xty)

    # TODO (b): hitung jarak (norma Euclidean) antara w_ridge dan bobot sebenarnya
    jarak = np.linalg.norm(w_ridge - ____)

    return {"lambda": lam, "w_ridge": np.round(w_ridge, 3), "jarak_ke_sebenarnya": jarak}


hasil_7 = [uji_lambda_ridge(lam) for lam in [1e-6, 1e-2, 1.0]]
tabel_7 = pd.DataFrame(hasil_7)
display(tabel_7)

### 7.2 Tabel Hasil Pengamatan

| λ | $\mathbf{w}_{\text{ridge}}$ | Jarak ke $[3,2,-1]$ |
|---|---|---|
| $10^{-6}$ | ...... | ...... |
| $10^{-2}$ | ...... | ...... |
| $1$ | ...... | ...... |

### 7.3 Pertanyaan Analisis

**7a.** Pada λ = $10^{-6}$ (regularisasi nyaris tidak ada), apakah bobotnya stabil atau justru mulai kacau seperti persamaan normal biasa pada Latihan 6?

> *Jawaban Anda:*
>
> ......

**7b.** Pada λ = 1 (regularisasi kuat), apakah bobotnya lebih stabil? Apakah jaraknya ke $[3,2,-1]$ menjadi lebih dekat atau lebih jauh dibandingkan λ = $10^{-2}$?

> *Jawaban Anda:*
>
> ......

**7c.** Simpulkan **pertukarannya**: apa yang dikorbankan untuk memperoleh kestabilan, dan apa yang dikorbankan untuk memperoleh kedekatan dengan bobot sebenarnya? Adakah λ yang menyeimbangkan keduanya pada percobaan Anda?

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 8 — Refleksi: Prediksi Akurat, Koefisien Tidak Dapat Dipercaya

**Soal asli:** *Jelaskan dengan kalimat Anda sendiri: mengapa dalam kasus ill-conditioning prediksi model masih bisa akurat, padahal koefisiennya sama sekali tidak dapat dipercaya?*

Latihan ini murni **konseptual** — tidak ada kode untuk dilengkapi. Namun sel berikut membantu Anda membuktikannya secara empiris sebelum menjawab.

In [ ]:
hasil_kacau = uji_kolinearitas(1e-8)
w_kacau = hasil_kacau["w_ne"]

# Membandingkan prediksi dari bobot yang "kacau" dengan bobot sebenarnya
rng_cek = np.random.default_rng(SEED + 1)
N_cek = 200
x1_cek = rng_cek.standard_normal(N_cek)
x2_cek = x1_cek + 1e-8 * rng_cek.standard_normal(N_cek)
X_cek = np.column_stack([np.ones(N_cek), x1_cek, x2_cek])

pred_kacau = X_cek @ w_kacau
pred_benar = X_cek @ np.array([3.0, 2.0, -1.0])

print("Bobot yang 'kacau'   :", np.round(w_kacau, 2))
print("Bobot sebenarnya     :", [3.0, 2.0, -1.0])
print("\nSelisih rata-rata prediksi (kacau vs sebenarnya):",
      round(np.mean(np.abs(pred_kacau - pred_benar)), 6))
print("Rentang nilai prediksi:", round(pred_benar.min(), 2), "sampai", round(pred_benar.max(), 2))

### 8.1 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| Bobot yang "kacau" (persamaan normal, gangguan $10^{-8}$) | ...... |
| Bobot sebenarnya | $[3,\ 2,\ -1]$ |
| Selisih rata-rata **prediksi** antara keduanya | ...... |

### 8.2 Pertanyaan Analisis

**8a.** Meskipun bobotnya sangat jauh berbeda, mengapa selisih rata-rata **prediksinya** tetap sangat kecil? Petunjuk: perhatikan bahwa $x_1$ dan $x_2$ pada data ini hampir identik.

> *Jawaban Anda:*
>
> ......

**8b.** Bila Anda seorang analis dan diminta menjelaskan "pengaruh $x_1$ terhadap $t$" berdasarkan model yang bobotnya kacau ini, apa risikonya bila Anda langsung menyebutkan angka koefisiennya?

> *Jawaban Anda:*
>
> ......

**8c.** Rumuskan jawaban akhir Anda untuk soal ini (Latihan Mandiri 8) dalam 3–5 kalimat, dengan menyebutkan istilah *span*, kolinearitas, dan identifiabilitas.

> *Jawaban Anda:*
>
> ......


---
# Kesimpulan dan Refleksi

### Kesimpulan

*Tuliskan minimal delapan poin kesimpulan — satu untuk setiap latihan mandiri — berdasarkan hasil percobaan Anda sendiri.*

1. (Geometri) ......
2. (Ridge) ......
3. (Lasso) ......
4. (Derajat vs λ) ......
5. (Koefisien nol) ......
6. (Kolinearitas vs bilangan kondisi) ......
7. (Kestabilan vs kedekatan) ......
8. (Prediksi vs koefisien) ......

### Refleksi

**R1.** Latihan mana yang paling mengubah pemahaman Anda tentang regularisasi? Jelaskan.

> *Jawaban Anda:*
>
> ......

**R2.** Error apa yang Anda temui saat mengerjakan, dan bagaimana Anda mengatasinya?

> *Jawaban Anda:*
>
> ......

**R3.** Bila diberi dataset nyata dengan gejala ill-conditioning, langkah apa saja yang akan Anda ambil berdasarkan pengalaman mengerjakan lembar kerja ini?

> *Jawaban Anda:*
>
> ......


---
# Rubrik Penilaian

*Bagian ini diisi oleh dosen atau asisten praktikum.*

| No | Aspek yang Dinilai | Bobot | Skor (0–100) | Nilai |
|---|---|---|---|---|
| 1 | Kelengkapan tabel hasil pengamatan (Latihan 1–8) | 20% | | |
| 2 | Ketepatan pengisian kode bertanda `TODO` | 25% | | |
| 3 | Kualitas jawaban analisis (1a–8c) | 35% | | |
| 4 | Kesimpulan dan refleksi | 20% | | |
| | **Nilai Akhir** | **100%** | | |

**Catatan dosen:**

> ......

### Pedoman skor jawaban analisis

| Skor | Kriteria |
|---|---|
| 85–100 | Jawaban tepat, didukung angka dari percobaan sendiri, dan menunjukkan penalaran yang jelas |
| 70–84 | Jawaban tepat tetapi kurang didukung data atau penalarannya dangkal |
| 55–69 | Jawaban sebagian benar, atau hanya mengulang teori tanpa mengaitkan hasil percobaan |
| < 55 | Jawaban tidak tepat, kosong, atau merupakan salinan dari mahasiswa lain |


---
# Daftar Periksa Sebelum Mengumpulkan

Centang dengan mengganti `[ ]` menjadi `[x]` (klik dua kali sel ini untuk mengedit).

- [ ] Identitas pada bagian atas sudah diisi lengkap
- [ ] `NIM_3_DIGIT` sudah diganti dengan NIM saya sendiri
- [ ] Semua sel bertanda `TODO` sudah dilengkapi dan berjalan tanpa error
- [ ] Seluruh sel sudah dijalankan berurutan dari atas ke bawah
- [ ] Semua tabel hasil pengamatan sudah diisi angka dari komputer saya
- [ ] Semua pertanyaan analisis 1a sampai 8c sudah dijawab
- [ ] Kesimpulan (8 poin) dan refleksi R1–R3 sudah ditulis
- [ ] Notebook sudah disimpan (`Ctrl + S`) sebelum diekspor

### Langkah ekspor PDF

1. Simpan notebook: `Ctrl + S`
2. `File` → `Save and Export Notebook As...` → `HTML`
3. Buka berkas HTML di browser, tekan `Ctrl + P`, pilih **Save as PDF**
4. Pada dialog cetak, aktifkan **Background graphics** agar grafik ikut tercetak berwarna
5. Beri nama berkas: `LKM_Latihan07_NIM_NamaLengkap.pdf`

---

*Lembar Kerja Mahasiswa — Latihan Mandiri untuk Pertemuan07_Regresi_Linear_2.ipynb*
*Materi diadaptasi dari kuliah "Linear Regression" oleh Joseph E. Gonzalez dan Narges Norouzi.*
